# One GRPO Step on One SWE-bench Task

**Goal: conceptual clarity, not training.** Nothing here converges. By the end
you will have watched a single gradient update travel the whole pipeline, with
every intermediate value printed.

| § | what it covers |
|---|---|
| 1 | one SWE-bench task, field by field |
| 2 | the environment actions land in — mocked, no Docker |
| 3 | the agent — `model + harness` |
| 4 | the reward, and why we cannot compute one |
| 5 | a group of rollouts |
| 6 | one GRPO update to the model |
| 7 | what the mocks cost |

### The one thing to hold onto

You can mock the filesystem. You can mock `grep`, `cat` and `pytest`. You
cannot mock the reward — a real one means really running the tests in a real
environment. §4 is where the pretending stops working, and §7 keeps the ledger
honest.

Runs on a T4 in a few minutes; on CPU if you're patient.

In [1]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch

# pick up edits to utils.py without restarting the kernel
%load_ext autoreload
%autoreload 2

import os, urllib.request
if not os.path.exists("utils.py"):      # Colab: fetch the plumbing
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py")

import json, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from utils import (TARGET, MOCK_FILE, MockEnv, SYSTEM, NO_COMMAND,
                   first_bash_block, generate, scripted_sampler, show_rollout)

for _opt in ["display.max_colwidth", "display.max_rows", "display.max_columns", "display.width"]:
    pd.set_option(_opt, None)

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

device: cuda


---
# 1. Load one instance

500 human-validated tasks. We take one and read it field by field — a SWE-bench
task is five artifacts, and only the first is the agent's input.

In [2]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
print(ds)

# A small, single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
         if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]
inst = ds[cands[0]]
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])
pass_to_pass = json.loads(inst["PASS_TO_PASS"])

display(pd.DataFrame(
    [(k, inst.get(k)) for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]],
    columns=["field", "value"],
))

display(pd.DataFrame([
    ("1.1", "problem_statement", "the agent",  "the input: a raw GitHub issue"),
    ("-",   "patch",             "nobody",     "reference solution; unused in this notebook"),
    ("1.2", "test_patch",        "the grader", "adds the tests that define 'fixed'"),
    ("1.3", "FAIL_TO_PASS",      "the grader", "must go red -> green"),
    ("1.4", "PASS_TO_PASS",      "the grader", "must stay green"),
], columns=["section", "field", "who sees it", "job"]))

/home/agoswami/miniconda3/envs/swe_grpo_vizuara_01/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty'],
    num_rows: 500
})


,field,value
0,instance_id,astropy__astropy-12907
1,repo,astropy/astropy
2,base_commit,d16bfe05a744909de4b27f5875fe0d4ed41ce607
3,version,4.3
4,difficulty,15 min - 1 hour


,section,field,who sees it,job
0,1.1,problem_statement,the agent,the input: a raw GitHub issue
1,-,patch,nobody,reference solution; unused in this notebook
2,1.2,test_patch,the grader,adds the tests that define 'fixed'
3,1.3,FAIL_TO_PASS,the grader,must go red -> green
4,1.4,PASS_TO_PASS,the grader,must stay green


## 1.1 `problem_statement` — the agent's entire input

A raw GitHub issue. The agent is never told which file to open, or that
`separable.py` exists. Finding it — localization — is most of the real
difficulty of SWE-bench.

In [3]:
print(inst["problem_statement"])

Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these compound models:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & cm)
array([[ True,  True, False, False],
       [ True,  True, False, 

## 1.2 `test_patch` — the grader's tests (held out, always applied)

The real PR changed two things: `separable.py` (the fix) and `test_separable.py`
(the regression tests). SWE-bench splits that PR — the source half becomes
`inst["patch"]`, which nothing here uses, and the test half becomes `test_patch`
— then pins `base_commit` to just before it.

So **these tests do not exist in the repository the agent works in.** Grading
has to inject them or nothing can detect the bug. The agent must not see them
either: the expected matrices are the answer.

In [4]:
print(inst["test_patch"])

diff --git a/astropy/modeling/tests/test_separable.py b/astropy/modeling/tests/test_separable.py
--- a/astropy/modeling/tests/test_separable.py
+++ b/astropy/modeling/tests/test_separable.py
@@ -28,6 +28,13 @@
 p1 = models.Polynomial1D(1, name='p1')
 
 
+cm_4d_expected = (np.array([False, False, True, True]),
+                  np.array([[True,  True,  False, False],
+                            [True,  True,  False, False],
+                            [False, False, True,  False],
+                            [False, False, False, True]]))
+
+
 compound_models = {
     'cm1': (map3 & sh1 | rot & sh1 | sh1 & sh2 & sh1,
             (np.array([False, False, True]),
@@ -52,7 +59,17 @@
     'cm7': (map2 | p2 & sh1,
             (np.array([False, True]),
              np.array([[True, False], [False, True]]))
-            )
+            ),
+    'cm8': (rot & (sh1 & sh2), cm_4d_expected),
+    'cm9': (rot & sh1 & sh2, cm_4d_expected),
+    'cm10': ((rot & sh1) & sh2, cm_4d_expected),
+    

## 1.3 `FAIL_TO_PASS` — must go red → green

The half of the reward that says *you solved the issue*.

The node ids look unrelated to the `cm*` keys above because `parametrize`
numbers cases positionally: `compound_models` holds 6 entries at `base_commit`
(`cm1`–`cm5`, `cm7`), and `test_patch` appends 4 more at indices 6–9.

The two that fail are exactly the **right-nested** ones — which is what the fix
repairs, in the `cright` branch.

In [5]:
for t in fail_to_pass:
    print(t)

new_cases = pd.DataFrame([
    (6, "cm8",  "rot & (sh1 & sh2)",         "right-nested"),
    (7, "cm9",  "rot & sh1 & sh2",           "flat"),
    (8, "cm10", "(rot & sh1) & sh2",         "left-nested"),
    (9, "cm11", "rot & sh1 & (scl1 & scl2)", "right-nested"),
], columns=["index", "case", "model", "grouping"])
new_cases.insert(1, "node_id", "compound_model" + new_cases["index"].astype(str))
new_cases["in_FAIL_TO_PASS"] = [
    any(f"[{n}-" in t for t in fail_to_pass) for n in new_cases["node_id"]
]
display(new_cases)

astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]


,index,node_id,case,model,grouping,in_FAIL_TO_PASS
0,6,compound_model6,cm8,rot & (sh1 & sh2),right-nested,True
1,7,compound_model7,cm9,rot & sh1 & sh2,flat,False
2,8,compound_model8,cm10,(rot & sh1) & sh2,left-nested,False
3,9,compound_model9,cm11,rot & sh1 & (scl1 & scl2),right-nested,True


## 1.4 `PASS_TO_PASS` — must stay green

The anti-reward-hacking half: without it, deleting the two failing tests would
score a win. SWE-bench also resets test files to `base_commit` before applying
`test_patch`, so tampering is discarded twice over.

Note that two of the four *new* cases land here, not in `FAIL_TO_PASS`. A new
test is not automatically a target — the lists are built by running the suite at
`base_commit` and sorting by observed outcome.

In [6]:
for t in pass_to_pass:
    print(t)

display(pd.DataFrame([
    ("existing cases, index 0-5",   6, "PASS_TO_PASS"),
    ("new, already green (7, 8)",   2, "PASS_TO_PASS"),
    ("new, red until fixed (6, 9)", 2, "FAIL_TO_PASS"),
    ("other tests in the file",     5, "PASS_TO_PASS"),
], columns=["group", "n_tests", "list"]))

print(f"\nFAIL_TO_PASS {len(fail_to_pass)}   PASS_TO_PASS {len(pass_to_pass)}")

astropy/modeling/tests/test_separable.py::test_coord_matrix
astropy/modeling/tests/test_separable.py::test_cdot
astropy/modeling/tests/test_separable.py::test_cstack
astropy/modeling/tests/test_separable.py::test_arith_oper
astropy/modeling/tests/test_separable.py::test_separable[compound_model0-result0]
astropy/modeling/tests/test_separable.py::test_separable[compound_model1-result1]
astropy/modeling/tests/test_separable.py::test_separable[compound_model2-result2]
astropy/modeling/tests/test_separable.py::test_separable[compound_model3-result3]
astropy/modeling/tests/test_separable.py::test_separable[compound_model4-result4]
astropy/modeling/tests/test_separable.py::test_separable[compound_model5-result5]
astropy/modeling/tests/test_separable.py::test_separable[compound_model7-result7]
astropy/modeling/tests/test_separable.py::test_separable[compound_model8-result8]
astropy/modeling/tests/test_separable.py::test_custom_model_separable


,group,n_tests,list
0,"existing cases, index 0-5",6,PASS_TO_PASS
1,"new, already green (7, 8)",2,PASS_TO_PASS
2,"new, red until fixed (6, 9)",2,FAIL_TO_PASS
3,other tests in the file,5,PASS_TO_PASS



FAIL_TO_PASS 2   PASS_TO_PASS 13


## 1.5 What the fields add up to

An issue, a fix, a test patch, two lists of names — and no repository. None of
it can be executed. Turning any of it into a number means cloning
`astropy/astropy` at `base_commit`, applying `test_patch`, applying the
candidate patch, and running both lists.

Those four steps *are* the reward function, and all four need a real
environment. We don't have one, so the next section builds a fake.

---
# 2. The environment

$$\text{state} \;\xrightarrow{\ \text{action}\ }\; \text{state}' \;+\; \text{observation}$$

- **state** — the files. Here, exactly one: `separable.py`.
- **action** — a bash command.
- **observation** — what the command printed. All the agent ever sees.

| § | |
|---|---|
| 2.1 | the initial state |
| 2.2 | take an action |
| 2.3 | the new state |

In production the environment is a container with the repo cloned at
`base_commit`. Ours is a Python dict, so `pytest` prints a canned failure and
never runs anything.

## 2.1 Initial state

One file, 29 lines — the real `_cstack` from `separable.py` at `base_commit`.
The bug is the asymmetry: `cleft` is assigned `left`, but `cright` is assigned
`1`.

In [7]:
env = MockEnv(fail_to_pass)

print("state = files in the environment:", list(env.fs), "\n")
print(env.fs[TARGET])

state = files in the environment: ['astropy/modeling/separable.py'] 

def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = 1

    return np.hstack([cleft, cright])


## 2.2 Take an action

An action is one bash command. The first three are read-only — they return an
observation and leave the state alone. Only the last one, a write, moves it.

In [8]:
FIXED = MOCK_FILE.replace("= 1", "= right")     # the one-token fix

rows = []
for action in ["ls",
               f'grep -n "cright" {TARGET}',
               "python -m pytest",
               f"cat > {TARGET} <<'EOF'\n{FIXED}\nEOF"]:
    before = dict(env.fs)
    obs = env.run(action)
    rows.append({"action":        action.splitlines()[0] + (" ..." if "\n" in action else ""),
                 "state changed": env.fs != before,
                 "observation":   obs.replace("\n", " | ")[:60]})

display(pd.DataFrame(rows))

,action,state changed,observation
0,ls,False,astropy/modeling/separable.py | setup.py | README.rst | test
1,"grep -n ""cright"" astropy/modeling/separable.py",False,"24: cright = _coord_matrix(right, 'right', noutp) | 2"
2,python -m pytest,False,[mock] FAILED astropy/modeling/tests/test_separable.py::test
3,cat > astropy/modeling/separable.py <<'EOF' ...,True,[mock] wrote 29 lines to astropy/modeling/separable.py


## 2.3 New state

The write landed. Here is the state now — and its difference from the initial
state **is** the candidate patch. Nothing else builds it.

In [9]:
print("new state:\n")
print(env.fs[TARGET])

print("\ninitial state vs new state — the candidate patch:\n")
print(env.patch())

new state:

def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = right

    return np.hstack([cleft, cright])

initial state vs new state — the candidate patch:

--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -24,6 +24,6 @@
         cright = _coord_matrix(ri

---
# 3. The agent

$$\textbf{agent} = \textbf{model} + \textbf{harness}$$

- **model** — text in, text out. Stateless. The only part that can learn.
- **harness** — deterministic code: prepare, parse, execute, append. Never trained.
- **rollout** — *not* a component. It is what you **get** when you run the agent
  once: one trajectory. `run_agent()` returns a rollout.

A model cannot open a file. A harness has nothing to say. Together they act on
the environment from §2.

| § | what we build |
|---|---|
| 3.1 | **the model** — Qwen2.5-Coder-0.5B, the policy |
| 3.2 | **the harness** — the loop, and the system prompt that fixes the action space |
| 3.3 | **one rollout** — the two together, run once |

## 3.1 The model

Qwen2.5-Coder-0.5B-Instruct — the policy. Nothing else: no adapter, because
running the agent needs no trainable parameters at all.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32,
    attn_implementation="sdpa").to(DEV)

print(f"{MODEL}\n{sum(p.numel() for p in model.parameters()):,} parameters, none trainable yet")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3133.18it/s]


Qwen/Qwen2.5-Coder-0.5B-Instruct
494,032,768 parameters, none trainable yet


## 3.2 The harness

```python
context = [SYSTEM, ISSUE]

for turn in range(MAX_TURNS):
    prompt = chat_template(context)      # 1. prepare    harness
    reply  = model.generate(prompt)      # 2. call       MODEL
    action = first_bash_block(reply)     # 3a. parse     harness
    obs    = env.run(action)             # 3b. execute   environment (§2)
    context += [reply, obs]              # 4. append     harness

return context                           # one rollout
```

One line is the model, one is the environment, the rest is harness — ordinary
code, with nothing learned in it.

Real harnesses add compaction and retrieval at step 1, permission gates at
step 3, and stop when the model asks for no more tools. We have none of that.

The action space lives in the system prompt — English prose, no tool schema:

In [11]:
print(SYSTEM)

You are a software engineering agent. You are in a Python repository.
Fix the bug described in the issue.

Respond with exactly ONE bash command per message, in a fenced block:

```bash
your command here
```

Useful commands:
  ls
  cat astropy/modeling/separable.py
  grep -n "pattern" astropy/modeling/separable.py
  python -m pytest

To rewrite a file:
```bash
cat > astropy/modeling/separable.py <<'EOF'
...full new contents...
EOF
```

One short sentence of reasoning, then exactly one command block.


The same loop in real code. `run_agent()` drives the model through the harness;
**one call returns one rollout** — the whole `context`, since in RL the
trajectory is the training example.

In [12]:
def run_agent(model, tok, max_turns=4, temperature=1.0, sample=generate):
    env = MockEnv(fail_to_pass)
    context = [{"role": "system", "content": SYSTEM},
               {"role": "user",   "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}]

    for _ in range(max_turns):
        prompt = tok.apply_chat_template(context, tokenize=False, add_generation_prompt=True)
        reply  = sample(model, tok, prompt, temperature=temperature)
        action = first_bash_block(reply)
        obs    = env.run(action) if action else NO_COMMAND
        context += [{"role": "assistant", "content": reply},
                    {"role": "user",      "content": obs[:800]}]

    return dict(messages=context, patch=env.patch(), final=dict(env.fs), calls=env.calls)

## 3.3 One rollout

Model + harness, run once against the environment. We do it twice: first with
the model we just loaded, then with a scripted stand-in — same harness, same
environment, only the model swapped.

### 3.3.1 With the model we loaded

Read the **written by** column: three parties author one trajectory, and only
the `model` rows are the policy's own output.

A 0.5B model almost never reaches a write, so the state never moves and the
patch comes out empty.

In [13]:
rollout = run_agent(model, tok)

In [14]:
show_rollout(rollout)

10 messages = 2 setup + 2 per turn x 4 turns



,i,turn,role,written by,chars,preview
0,0,setup,system,harness,505,You are a software engineering agent. You are in a Python repository. |
1,1,setup,user,harness,1253,ISSUE: | Modeling's `separability_matrix` does not compute separability
2,2,0,assistant,model,75,```bash | ls | grep -n 'astropy.modeling.separable.py' | awk '{print $1}
3,3,0,user,environment,56,astropy/modeling/separable.py | setup.py | README.rst | tests/
4,4,1,assistant,model,35,```bash | python setup.py install | ```
5,5,1,user,environment,44,[mock] 'python' not implemented in this stub
6,6,2,assistant,model,38,```bash | mock -m unittest.mock -v 3 | ```
7,7,2,user,environment,42,[mock] 'mock' not implemented in this stub
8,8,3,assistant,model,38,```bash | mock -m unittest.mock -v 3 | ```
9,9,3,user,environment,42,[mock] 'mock' not implemented in this stub



commands:
  $ ls | grep -n 'astropy.modeling.separable.py' | awk '{print $1}'
  $ python setup.py install
  $ mock -m unittest.mock -v 3
  $ mock -m unittest.mock -v 3

candidate patch:
(no file changed)


### 3.3.2 With a scripted model

`run_agent`'s `sample` argument swaps the model and nothing else — same harness,
same environment. The replies below are canned, not generated.

Hold the harness fixed, change the model, get a different rollout:
`agent = model + harness`, made literal.

In [15]:
SCRIPT = [
    f"Let me read the file.\n```bash\ncat {TARGET}\n```",
    f"The cright branch assigns 1 instead of right. Fixing it.\n"
    f"```bash\ncat > {TARGET} <<'EOF'\n{FIXED}\nEOF\n```",
    "Now run the tests.\n```bash\npython -m pytest\n```",
    "Check the tree.\n```bash\nls\n```",
]

expert = run_agent(model, tok, sample=scripted_sampler(SCRIPT))

In [16]:
show_rollout(expert)

10 messages = 2 setup + 2 per turn x 4 turns



,i,turn,role,written by,chars,preview
0,0,setup,system,harness,505,You are a software engineering agent. You are in a Python repository. |
1,1,setup,user,harness,1253,ISSUE: | Modeling's `separability_matrix` does not compute separability
2,2,0,assistant,model,67,Let me read the file. | ```bash | cat astropy/modeling/separable.py | ```
3,3,0,user,environment,800,"[mock: only the region near the bug is available] | def _cstack(left, ri"
4,4,1,assistant,model,928,The cright branch assigns 1 instead of right. Fixing it. | ```bash | cat >
5,5,1,user,environment,54,[mock] wrote 29 lines to astropy/modeling/separable.py
6,6,2,assistant,model,47,Now run the tests. | ```bash | python -m pytest | ```
7,7,2,user,environment,237,[mock] FAILED astropy/modeling/tests/test_separable.py::test_separable
8,8,3,assistant,model,30,Check the tree. | ```bash | ls | ```
9,9,3,user,environment,56,astropy/modeling/separable.py | setup.py | README.rst | tests/



commands:
  $ cat astropy/modeling/separable.py
  $ cat > astropy/modeling/separable.py <<'EOF' ...
  $ python -m pytest
  $ ls

candidate patch:
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -24,6 +24,6 @@
         cright = _coord_matrix(right, 'right', noutp)
     else:
         cright = np.zeros((noutp, right.shape[1]))
-        cright[-right.shape[0]:, -right.shape[1]:] = 1
+        cright[-right.shape[0]:, -right.shape[1]:] = right
 
     return np.hstack([cleft, cright])


---
# 4. Reward

Here is where mocking runs out.

**The real reward** clones the repo at `base_commit`, applies `test_patch`,
applies the candidate patch, runs `FAIL_TO_PASS` and `PASS_TO_PASS`, and returns
`1.0` only if every test in both lists passes. Binary, no partial credit.

We cannot do that offline, and there is no honest shortcut — whether a patch is
correct is not a property of its text. So we don't pretend: the reward below is
**random numbers**. It says nothing about the rollouts.

Every other mock in this notebook costs fidelity. This one removes the signal
entirely.

In [17]:
def reward_random(n, seed=0):
    """Stand-in for a real verifier (unit tests, or a reward model).

    One value per rollout, seeded for reproducibility. Carries NO information
    about rollout quality -- it only lets us show how the update consumes it.
    """
    return np.random.default_rng(seed).random(n).round(3)

print(reward_random(6))

[0.637 0.27  0.041 0.017 0.813 0.913]


---
# 5. Sample a group

§3.3 ran the agent once. GRPO needs several rollouts of the **same** task to
compare against each other, so now we run it `G = 6` times from the same prompt
at `temperature=1.0`. The spread between them is the only thing the update has
to work with.

Same harness, same environment, same prompt every time — the only thing that
varies is the model's sampling. A 0.5B model is well below the level this task
needs, so the traces below are mostly nonsense: invented commands, prose where a
command should be. That is worth seeing rather than hiding.

In [18]:
G = 6
group = [run_agent(model, tok) for _ in range(G)]
reward = reward_random(len(group))

display(pd.DataFrame([
    {
        "rollout":     i,
        "reward":      reward[i],
        "patch_lines": len(r["patch"].split("\n")) if r["patch"] else 0,
        "n_commands":  len(r["calls"]),
    }
    for i, r in enumerate(group)
]))

for i, r in enumerate(group):
    print(f"\n================ rollout {i} ================")
    for cmd in r["calls"]:
        print("  $", cmd)
    print("\ncandidate patch:")
    print(r["patch"] if r["patch"] else "(no file changed)")

,rollout,reward,patch_lines,n_commands
0,0,0.637,0,1
1,1,0.270,0,3
2,2,0.041,0,4
3,3,0.017,0,3
4,4,0.813,47,3
5,5,0.913,0,4



================ rollout 0 ================
  $ # Assuming 'Linear1D' and 'Pix2Sky_TAN' are defined earlier in your script
# Convert Linear1D instances to a list of models
linear_models = list(Linear1D(x * 0.1) for x in range(10))

# Compute the separability matrix for each list separately
for i, model in enumerate(linear_models):
    matrix = separability_matrix(model)
    print(f"Separability Matrix for model {i+1}: {matrix}")

# Now the inputs and outputs are separable

candidate patch:
(no file changed)

================ rollout 1 ================
  $ ls
  $ ls tests/README.rst
  $ ls tests

candidate patch:
(no file changed)

================ rollout 2 ================
  $ # Use 'grep' to find the line that contains the pattern
echo 'grep -n "pattern" astropy/modeling/separable.py' | bash
  $ # Run mock to test with input
mock astropy/modeling/separable.py
  $ # Remove '#' if it was there
sed 's/^#\(.*\)/#/' astropy/modeling/separable.py
  $ # Check the contents of astropy/modeli

## 5.1 Degenerate groups

GRPO's baseline is the group itself, so a group whose rollouts all score the
same has zero spread, zero advantage, and contributes zero gradient.

With a real binary reward that is the *normal* outcome here: a 0.5B model solves
none of the six, every reward is `0.0`, and the step is a no-op. Keeping groups
non-degenerate — task difficulty, temperature, group size — is most of the
practical work in GRPO. Our random reward sidesteps it by construction.

In [19]:
flat = np.full(len(group), 0.5)

display(pd.DataFrame({
    "group":        ["reward (this run)", "if all rollouts tied"],
    "values":       [str(reward), str(flat)],
    "std":          [reward.std().round(4), flat.std().round(4)],
    "any gradient": [reward.std() > 1e-6, flat.std() > 1e-6],
}))

,group,values,std,any gradient
0,reward (this run),[0.637 0.27 0.041 0.017 0.813 0.913],0.3578,True
1,if all rollouts tied,[0.5 0.5 0.5 0.5 0.5 0.5],0.0000,False


---
# 6. One GRPO step

$$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r) + \varepsilon}$$

No critic. **The other rollouts are the baseline** — that is the entire idea.
Loss is a policy gradient over assistant tokens only; observation tokens came
from the environment, so crediting them is meaningless.

Only now do we need trainable parameters, so only now do we add LoRA. It is
initialised to the identity, which means the policy that produced the rollouts
in §5 is exactly the policy we are about to update — the update is on-policy.

In [20]:
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [21]:
def build_masked(messages, tokenizer, max_len=3072):
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i+1], tokenize=False)
        assert cur.startswith(prev), "chat template is not append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids += seg
        labels += seg if m["role"] == "assistant" else [-100]*len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def seq_logprob(messages):
    ids, labs = build_masked(messages, tok)
    t = torch.tensor([ids], device=DEV)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=DEV)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * msk).sum() / msk.sum().clamp(min=1), msk.sum().item()

In [22]:
rewards = torch.tensor(reward, dtype=torch.float)
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-4)

print(f"mean reward {rewards.mean():.3f}   std {rewards.std(unbiased=False):.3f}")
display(pd.DataFrame({
    "rollout":    list(range(len(group))),
    "reward":     rewards.numpy().round(3),
    "advantage":  adv.numpy().round(3),
    "sup_tokens": [int(seq_logprob(g["messages"])[1]) for g in group],
}))

mean reward 0.448   std 0.358


,rollout,reward,advantage,sup_tokens
0,0,0.637,0.527,426
1,1,0.270,-0.499,253
2,2,0.041,-1.138,135
3,3,0.017,-1.206,385
4,4,0.813,1.018,560
5,5,0.913,1.298,85


In [23]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

logp_before = [seq_logprob(g["messages"])[0].item() for g in group]

model.train(); opt.zero_grad(set_to_none=True)
loss_total = 0.0
for g, a in zip(group, adv):
    lp, _ = seq_logprob(g["messages"])
    loss = -(a.to(DEV) * lp) / len(group)      # REINFORCE with a group baseline
    loss.backward()
    loss_total += loss.item()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad], 1.0)
opt.step()

logp_after = [seq_logprob(g["messages"])[0].item() for g in group]

print(f"loss {loss_total:+.5f}   grad_norm {grad_norm:.4f}")
display(pd.DataFrame({
    "rollout":     list(range(len(group))),
    "advantage":   [round(a.item(), 3) for a in adv],
    "logp_before": [round(b, 4) for b in logp_before],
    "logp_after":  [round(c, 4) for c in logp_after],
    "delta":       [round(c - b, 5) for b, c in zip(logp_before, logp_after)],
}))

loss -0.05272   grad_norm 0.6246


,rollout,advantage,logp_before,logp_after,delta
0,0,0.527,-1.1735,-1.1730,0.00057
1,1,-0.499,-1.1900,-1.1891,0.00089
2,2,-1.138,-1.7507,-1.7508,-0.00011
3,3,-1.206,-1.4771,-1.4769,0.00017
4,4,1.018,-0.6781,-0.6776,0.00047
5,5,1.298,-2.1133,-2.1084,0.00490


**The mechanism.** Positive advantage pushes a trajectory's log-probability up,
negative pushes it down. No value network, no reward model — just *this
trajectory scored better than its siblings, so make it more likely.*

**Why the `delta` column does not show that yet.** On AdamW's first step the
moment estimates are empty, so every parameter moves by exactly `lr` in the
direction of its gradient's *sign*, regardless of magnitude. All six
trajectories share one LoRA, so a single update drags them together, and at
`lr=1e-5` that shared movement is larger than the differences between them. One
step is not enough to separate the signs; many steps are.

Worth saying plainly to the room: a mechanism can be implemented correctly and
still be invisible in a one-step demo.

---
# 7. The honest ledger

What we faked, and what each shortcut cost:

| Faked | Real version | What the fake hides |
|---|---|---|
| repo → one hardcoded function | full clone at `base_commit` | the agent can't explore, so localization — most of the real difficulty — vanishes |
| `cat`/`grep` → dict lookup | shell in a container | no build, no imports, no cross-file reasoning |
| `pytest` → canned string | real suite at real commit | **everything**; see below |
| reward → random numbers | `FAIL_TO_PASS ∧ PASS_TO_PASS` | there is no learning signal at all |
| 1 task | 500 (Verified) / 50k (SWE-smith) | no generalization claim is possible |
| 1 step | thousands | no learning |

The first three cost **fidelity** — a worse environment, but still an
environment. The fourth is different in kind. Sections 5 and 6 ran real
arithmetic on meaningless numbers: real rollouts, a real advantage
calculation, a real gradient, and a real optimizer step, all driven by
`np.random`. Every mechanism worked; the loop learned nothing, and could not.

This is why production SWE RL spends its budget on container fleets, and why
the verifier — not the GPU — is usually the bottleneck.

Good closing question for the room: *every number in §6 was computed correctly,
and the run was still worthless. What is the smallest change that would fix
that?*

---
## Where to go next

- **Lab 1** — trajectory SFT, loss masking, real SWE-smith data
- **Lab 2** — a full GRPO loop with real execution rewards, on a shrunken gym
- Real frameworks: `NovaSky-AI/SkyRL`, `PrimeIntellect-ai/prime-rl`
- The harness this imitates: `SWE-agent/mini-swe-agent` (pin `<2`)